In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\Customer Shopping Behavior Analysis\data\customer_shopping_behavior.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(lambda x : x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
df = df.rename(columns={'Purchase Amount (USD)' : 'Purchase Amount'})
df.columns

In [ ]:
labels = ["Young Adult" , "Adult" , "Middle Age" , "Senior"]
df['Age_group'] = pd.qcut(df['Age'] , q = 4 , labels = labels)

In [ ]:
df[["Age" ,"Age_group"]].head()

In [ ]:
# Renaming columns according to snake casing for better readability and documentation

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
df.columns

In [ ]:
#  create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

In [ ]:
df[['discount_applied' , 'promo_code_used']].head(10)

In [ ]:
df['discount_applied'] == df['promo_code_used']

In [ ]:
df = df.drop('promo_code_used' , axis = 1)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Load .env file from project root
env_path = Path.cwd() / ".env"

# If notebook/file is running from a subfolder, go one level back
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

# Load .env and override old cached values
load_dotenv(env_path, override=True)

# MySQL connection details from .env
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT", "3306")
database = os.getenv("DB_NAME")

if not all([username, password, host, port, database]):
    raise ValueError("Database credentials missing. Please check your .env file.")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=host,
    port=int(port),
    database=database
)

engine = create_engine(connection_url)

print("MySQL engine created successfully")

In [ ]:
# Write DataFrame to MySQL
table_name = "customer"

df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)